# Imports

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# Locate Files

In [ ]:
data_folder = Path("/your/path/to/mat/files")

mat_files = sorted(data_folder.glob("*.mat"))

print(f"Found {len(mat_files)} MATLAB files\n")

for file in mat_files:
    print(file.name)

# Verify patient IDs and channel availability

In [ ]:
selected_channels = ["reref67", "reref56"]
method = "rollmed"

for file in mat_files:

    with h5py.File(file, "r") as data:

        patients = [
            key for key in data.keys()
            if not key.startswith("#")
            and isinstance(data[key], h5py.Group)
            and "dbs/singles/bipolar" in data[key]
        ]

        print(f"\nFile: {file.name}")

        for patient in patients:

            print(f"Patient: {patient}")

            for channel in selected_channels:

                path = (
                    f"{patient}/dbs/singles/"
                    f"bipolar/{channel}/abs"
                )

                if path in data:

                    amplitudes = list(data[path].keys())

                    available = [
                        amp for amp in amplitudes
                        if method in data[path][amp]
                    ]

                    print(f"  {channel}: {len(available)} amplitudes")
                    print(f"  {available}")

                else:
                    print(f"  WARNING: {channel} not found")

# Extraction Function

In [ ]:
def extract_erna_data(
    mat_files,
    output_path,
    channels=("reref67", "reref56"),
    method="rollmed",
    waveform_dtype=np.float32,
    batch_trials=16
):

    summary = []

    # Create new output HDF5 file
    with h5py.File(output_path, "x") as output:

        output.attrs["preprocessing"] = method
        output.attrs["reference_type"] = "bipolar"

        # Iterate through patient files
        for file in mat_files:

            with h5py.File(file, "r") as data:

                # Automatically identify patient
                patients = [
                    key for key in data.keys()
                    if not key.startswith("#")
                    and isinstance(data[key], h5py.Group)
                    and "dbs/singles/bipolar" in data[key]
                ]

                if len(patients) != 1:
                    raise ValueError(
                        f"{file.name}: expected one patient, "
                        f"found {len(patients)}"
                    )

                patient = patients[0]

                print(f"\nProcessing patient: {patient}")

                patient_group = output.create_group(patient)
                patient_group.attrs["source_file"] = file.name

                # Iterate through selected channels
                for channel in channels:

                    source_path = (
                        f"{patient}/dbs/singles/"
                        f"bipolar/{channel}/abs"
                    )

                    if source_path not in data:
                        print(f"WARNING: Missing {channel}")
                        continue

                    source_channel = data[source_path]

                    channel_group = patient_group.create_group(
                        channel
                    )

                    # Iterate through all available amplitudes
                    for amplitude in source_channel.keys():

                        condition = source_channel[amplitude]

                        if not isinstance(condition, h5py.Group):
                            continue

                        if method not in condition:
                            print(
                                f"WARNING: Missing "
                                f"{channel}/{amplitude}/{method}"
                            )
                            continue

                        source = condition[method]
                        meta = condition["meta"]

                        # Extract numerical metadata
                        amp_abs = meta["amp_abs"][()].item()
                        fs = source["fs"][()].item()

                        # Extract time and mean waveform
                        t_ms = source["t_ms"][:].squeeze()
                        mean_wave = source["mean_wave"][:].squeeze()

                        # Access trials without loading them yet
                        source_trials = source["trials"]

                        n_samples, n_trials = source_trials.shape

                        # Verify dimensions and metadata
                        if n_samples != t_ms.size:
                            raise ValueError(
                                "Trial and time dimensions do not match"
                            )

                        if mean_wave.size != n_samples:
                            raise ValueError(
                                "Mean waveform and time dimensions "
                                "do not match"
                            )

                        if int(meta["n_trials"][()].item()) != n_trials:
                            raise ValueError(
                                "Trial count does not match metadata"
                            )

                        # Create amplitude group
                        target = channel_group.create_group(amplitude)

                        target.attrs["amp_abs"] = amp_abs
                        target.attrs["fs"] = fs
                        target.attrs["n_trials"] = n_trials

                        # Save time vector
                        target.create_dataset(
                            "t_ms",
                            data=t_ms,
                            dtype=np.float64
                        )

                        # Save mean waveform
                        target.create_dataset(
                            "mean_wave",
                            data=mean_wave,
                            dtype=waveform_dtype,
                            compression="lzf"
                        )

                        # Create empty trials dataset
                        target_trials = target.create_dataset(
                            "trials",
                            shape=(n_samples, n_trials),
                            dtype=waveform_dtype,
                            chunks=(n_samples, 1),
                            compression="lzf",
                            shuffle=True
                        )

                        # Copy trials in small batches
                        for start in range(0, n_trials, batch_trials):

                            stop = min(
                                start + batch_trials,
                                n_trials
                            )

                            target_trials[:, start:stop] = (
                                source_trials[:, start:stop]
                                .astype(waveform_dtype)
                            )

                        # Record extraction information
                        summary.append({
                            "patient": patient,
                            "channel": channel,
                            "amplitude_key": amplitude,
                            "amp_abs": amp_abs,
                            "method": method,
                            "fs": fs,
                            "n_samples": n_samples,
                            "n_trials": n_trials
                        })

                        print(
                            f"  {channel} | "
                            f"Amp: {amp_abs} | "
                            f"Trials: {n_trials}"
                        )

    print("\nExtraction complete!")

    return pd.DataFrame(summary)

# Extract All 6 Patients

In [ ]:
output_path = data_folder / "ERNA_extracted.h5"

summary = extract_erna_data(
    mat_files=mat_files,
    output_path=output_path,
    channels=["reref67", "reref56"],
    method="rollmed",
    waveform_dtype=np.float32,
    batch_trials=16
)

# Display Summary

In [ ]:
display(summary)

# Patient and Trial Counts

In [ ]:
print("Number of patients:", summary["patient"].nunique())
print("Number of conditions:", len(summary))
print("Total channel-trial waveforms:", summary["n_trials"].sum())

print("\nPatients:")
print(summary["patient"].unique())

print("\nConditions per patient and channel:")

display(
    summary.groupby(["patient", "channel"])
    .size()
    .unstack(fill_value=0)
)

# Amplitude availability by patient and channel

In [ ]:
print("Number of patients:", summary["patient"].nunique())
print("Number of conditions:", len(summary))
print("Total channel-trial waveforms:", summary["n_trials"].sum())

print("\nPatients:")
print(summary["patient"].unique())

print("\nConditions per patient and channel:")

display(
    summary.groupby(["patient", "channel"])
    .size()
    .unstack(fill_value=0)
)

# Save Summary as csv

In [ ]:
summary.to_csv(
    data_folder / "ERNA_extraction_summary.csv",
    index=False
)

# Reusable loading function

In [ ]:
def load_erna(
    file_path,
    patient,
    channel,
    amplitude,
    load_trials=False
):

    with h5py.File(file_path, "r") as data:

        group = data[f"{patient}/{channel}/{amplitude}"]

        result = {
            "patient": patient,
            "channel": channel,
            "amp_abs": group.attrs["amp_abs"],
            "fs": group.attrs["fs"],
            "n_trials": group.attrs["n_trials"],
            "t_ms": group["t_ms"][:],
            "mean_wave": group["mean_wave"][:]
        }

        if load_trials:
            result["trials"] = group["trials"][:]

    return result

# Example of using the above function

In [ ]:
recording = load_erna(
    file_path=output_path,
    patient="ug3008",
    channel="reref67",
    amplitude="amp2",
    load_trials=True
)